In [ ]:
!pip install ultralytics roboflow -q

from roboflow import Roboflow
import torch
from getpass import getpass

# Verificar GPU disponible
print(f"GPU disponible: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

from roboflow import Roboflow
rf = Roboflow(api_key="(busquen su API key en https://roboflow.com/account)")
project = rf.workspace("edwins-workspace-eqowv").project("cacahuate-v2")
version = project.version(1)
dataset = version.download("yolov8")
# Guardar la ruta del data.yaml para usarla después
import os
DATA_YAML = os.path.join(dataset.location, "data.yaml")
print(f"✅ Dataset listo en: {dataset.location}")
print(f"📄 data.yaml en: {DATA_YAML}")

# Siempre verifica qué clases tiene tu dataset antes de entrenar
with open(DATA_YAML, "r") as f:
    print(f.read())


from ultralytics import YOLO

# Modelo base: yolov8m-seg (medium, segmentación)
# Se descarga automáticamente ~52MB de pesos pre-entrenados en COCO
model = YOLO("yolov8m-seg.pt")

results = model.train(
    # ── Dataset ──────────────────────────────────────────
    data=DATA_YAML,

    # ── Épocas y early stopping ───────────────────────────
    epochs=200,           # Máximo de épocas
    patience=20,          # Early stopping: para si no mejora en 30 épocas seguidas

    # ── Imagen y batch ────────────────────────────────────
    imgsz=640,            # Tamaño estándar YOLOv8
    batch=64,

    # ── Optimizador ───────────────────────────────────────
    optimizer="AdamW",    # Más estable que SGD para datasets medianos
    lr0=0.001,            # Learning rate inicial
    lrf=0.01,             # Factor de decay al final del entrenamiento
    weight_decay=0.0005,
    warmup_epochs=5,

    # ── Hardware ──────────────────────────────────────────
    device=0,             # GPU 0 de Colab
    workers=4,            # Hilos de carga de datos
    cache=True,           # Cachea imágenes en RAM (acelera tras la 1ra época)

    # ── Guardar ───────────────────────────────────────────
    name="carril_seg_m",  # Nombre del experimento
    save=True,
    save_period=10,       # Guarda checkpoint cada 10 épocas (por si Colab desconecta)

    # ── Logs ──────────────────────────────────────────────
    plots=True,           # Genera gráficas de loss, mAP, etc.
    verbose=True,
)

print("\n✅ Entrenamiento terminado")
print(f"📁 Resultados en: runs/segment/carril_seg_m/")


# Evaluar el mejor modelo guardado
best_model = YOLO("runs/segment/carril_seg_m/weights/best.pt")

metrics = best_model.val(data=DATA_YAML, imgsz=640)

print(f"\n📈 MÉTRICAS FINALES")
print(f"  mAP50       (box):  {metrics.box.map50:.4f}")
print(f"  mAP50-95    (box):  {metrics.box.map:.4f}")
print(f"  mAP50       (mask): {metrics.seg.map50:.4f}")   # ← esta es la importante
print(f"  mAP50-95    (mask): {metrics.seg.map:.4f}")     # ← y esta

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 89.5 MB/s eta 0:00:00
GPU disponible: True
GPU: NVIDIA A100-SXM4-40GB
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to cacahuate-v2-1 in yolov8:: 100%|██████████| 16724/16724 [00:02<00:00, 7236.11it/s] 


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Dataset listo en: /content/cacahuate-v2-1
📄 data.yaml en: /content/cacahuate-v2-1/data.yaml
names:
- carril
nc: 1
roboflow:
  license: CC BY 4.0
  project: cacahuate-v2
  url: https://universe.roboflow.com/edwins-workspace-eqowv/cacahuate-v2/dataset/1
  version: 1
  workspace: edwins-workspace-eqowv
test: ../test/images
train: ../train/images
val: ../valid/images

Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None